## DATA ANALYSIS FOR THE ML PROJECT

# Importing and display of the data


In [1]:
import numpy as np
import implementations
import csv

In [ ]:
x_train = np.genfromtxt("x_train.csv", delimiter=",", names = True) 
y_train = np.genfromtxt("y_train.csv", delimiter=",", names = True)

In [ ]:
headers = list(x_train.dtype.names) #Get the names of the variables taken into account in the dataset, aswell as the shape of the dataset.
num_columns = len(headers)
num_rows = x_train.shape[0]

print (f"Headers: {headers}")
print (f"Number of columns: {num_columns}")
print (f"Number of rows: {num_rows}")


In [ ]:
for row in (x_train[:5]): #Display the first rows of the dataset to get a better understanding of the data.
    print (row)

# Cleaning of the dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.DataFrame(x_train)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Heatmap showing exact locations of NaNs across rows & columns
sns.heatmap(df.isnull(), cbar=False, cmap="binary", yticklabels=False, ax=axes[0])
axes[0].set_title("NaN Distribution Across Dataset (Black = Missing)")
axes[0].set_xlabel("Features")
axes[0].set_ylabel("Rows / Samples")

# 2. Bar plot showing percentage of NaNs per feature
nan_percent = (df.isnull().sum() / len(df)) * 100
nan_percent = nan_percent[nan_percent > 0].sort_values(ascending=False)

if not nan_percent.empty:
    sns.barplot(x=nan_percent.values, y=nan_percent.index, palette="viridis", ax=axes[1])
    axes[1].set_title("Percentage of Missing Values per Feature")
    axes[1].set_xlabel("% Missing")
    axes[1].set_ylabel("Features")
    axes[1].set_xlim(0, 100)
    
    # Add value labels to bars
    for index, value in enumerate(nan_percent.values):
        axes[1].text(value + 1, index, f"{value:.1f}%", va='center')
else:
    axes[1].text(0.5, 0.5, "No missing values found!", ha='center', va='center', fontsize=12)

plt.tight_layout()
plt.show()

In [32]:
nan_percentages = {} #Clean the dataset by checking for columns with a high percentage of NaN values and removing them from the dataset.

for col in headers:
    nan_count = np.isnan(x_train[col]).sum()
    nan_percentage = (nan_count / num_rows)
    nan_percentages[col] = nan_percentage


high_nan_columns = [col for col, pct in nan_percentages.items() if pct > 0.5] #List of the high percentage NaN columns.
print (f"The number of columns with more than half percent of Nan values is {len(high_nan_columns)}")
                    
cols_to_keep = [col for col in headers if col not in high_nan_columns]  
x_train_clean = np.column_stack([x_train[col] for col in cols_to_keep])

print (f"The shape of the cleaned dataset is {x_train_clean.shape}")

The number of columns with more than half percent of Nan values is 147
The shape of the cleaned dataset is (328135, 175)


In [34]:
#Now we will check for the remaining NaN values in the dataset and replace them with the mean of each column.

for i in range (len(cols_to_keep)):
    col_data = x_train_clean[:, i]
    nan_left = np.isnan(col_data)
    if nan_left.any():
       col_mean_value = np.nanmean(col_data)
       col_data[nan_left] = col_mean_value


total_nan_left = np.isnan(x_train_clean).sum()
print (f"The total NaN values lefet in the dataset is {total_nan_left}")


       

The total NaN values lefet in the dataset is 0


In [43]:
mean = np.mean (x_train_clean, axis = 0)
std = np.std (x_train_clean, axis = 0)

std[std == 0] = 1

x_train_normalized = (x_train_clean - mean) / std

num_cols_clean = x_train_normalized.shape[1]
num_rows_clean = x_train_normalized.shape[0]


print (num_cols_clean, num_rows_clean)

175 328135


# Studying the colinearity of the data


In [ ]:
corr_matrix = np.corrcoef(x_train_normalized, rowvar=False)

high_correlated_pairs = []

for i in range (num_cols_clean):
    for j in range (i + 1, num_cols_clean):
        if abs(corr_matrix[i, j]) > 0.9:
            high_correlated_pairs.append((cols_to_keep[i], cols_to_keep[j], corr_matrix[i, j]))

print (f"The variable pairs with a high correlation are: ", end = "\n")
for pair in high_correlated_pairs:
    print (f"{pair[0]} and {pair[1]}: {pair[2]:.3f}")